## SJM with Nudged Elastic Band


This notebook demonstrates the combined SJM-NEB workflow for H atom diffusion on Pt(111) from an ontop site to a hollow site in the presence of a water layer. The initial and final states are separately optimized at the target potential of 4.2 eV work function using a loose first pass with always_adjust=True (allowing ionic and electronic degrees of freedom to relax simultaneously) followed by a tight second pass. Five intermediate images are then interpolated, each given its own SJM calculator initialized with an excess_electrons and slope guess from the converged endpoint, and the band is relaxed with the Dynamic NEB algorithm.

In [1]:
%%capture
!apt install libxc-dev
!pip install gpaw
!echo "y" | gpaw install-data /usr/local/pkgs/gpaw-data-0.9.20000/

In [2]:
!pip install -q gdown
!pip -q install py3Dmol

In [3]:
from gpaw import setup_paths
setup_paths.insert(0, '/usr/local/pkgs/gpaw-data-0.9.20000-hd8ed1ab_2/share/gpaw/')

In [4]:
from ase import Atom
import ase.io
from ase.build import fcc111, molecule
from ase.units import Pascal, m
from ase.optimize import BFGS
from ase.constraints import FixAtoms
from ase.mep import interpolate, DyNEB, NEBTools
from ase.visualize import view
from ase.io import write, read
from ase.io.cube import read_cube_data, write_cube
import py3Dmol
import numpy as np
from matplotlib import pyplot
import os

from gpaw.solvation.sjm import SJM, SJMPower12Potential
from gpaw.solvation import (
    EffectivePotentialCavity,
    LinearDielectric,
    GradientSurface,
    SurfaceInteraction)

Build a Pt slab, put a water molecule on top and calculate systems' potential energy

Optimized structure:

In [5]:
!gdown --fuzzy "https://drive.google.com/file/d/1JaonEpy0oLvKI8DDKOLfsoyjUd2o9IYT/view?usp=drive_link" -O Pt111-water.xyz
atoms = ase.io.read('Pt111-water.xyz')
view(atoms, viewer='x3d')

Downloading...
From: https://drive.google.com/uc?id=1JaonEpy0oLvKI8DDKOLfsoyjUd2o9IYT
To: /content/Pt111-water.xyz
100% 834/834 [00:00<00:00, 2.86MB/s]


We load the pre-optimized Pt(111) slab with water molecules from the previous notebook and add a hydrogen atom above an ontop site. This will be our initial state. To speed up convergence, we first run a loose optimization where the potential tolerance is relaxed and `always_adjust` is set to True, allowing ionic and electronic degrees of freedom to relax simultaneously. We then tighten the tolerances in a second pass to reach a well-converged structure. Note that this calculation takes approximately 1 hour and 20 minutes.

In [6]:
# Add an H adsorbate.
atoms.append(Atom('H', atoms[2].position + (0., 0., 1.5)))
view(atoms, viewer='x3d')

~1h20 min - relaxing the slab and adsorbate  
Add an H atom to the Pt slab with a water molecule surface as an adsorbate.


It is often faster to optimize the structure and the potential simultaneously. That is, instead of waiting until the potential is perfectly equilibrated before taking an ionic step, you can take an ionic step and adjust the number of electrons simultaneously. To do this, set a loose potential tolerance and turn the **always_adjust** keyword to **True**. During the run potential may change but in the last iterations will be kept constant.

In [ ]:
# Fix some atoms.
atoms.set_constraint(FixAtoms(indices=[0, 1]))
view(atoms, viewer='x3d')

# Solvated jellium parameters.
sj = {'target_potential': 4.2,
      'tol': 0.2,
      'always_adjust': True}

# Implicit solvent parameters (to SolvationGPAW).
epsinf = 78.36  # dielectric constant of water at 298 K
gamma = 18.4 * 1e-3 * Pascal * m
cavity = EffectivePotentialCavity(
    effective_potential=SJMPower12Potential(H2O_layer=False),
    temperature=298.15,  # K
    surface_calculator=GradientSurface())
dielectric = LinearDielectric(epsinf=epsinf)
interactions = [SurfaceInteraction(surface_tension=gamma)]

# The calculator
calc = SJM(
    # General GPAW parameters.
    mode='lcao',
    txt='Pt111-H-sim.txt',
    gpts=(16, 16, 136),
    kpts=(2, 2, 1),
    xc='PBE',
    maxiter=1000,
    # Solvated jellium parameters.
    sj=sj,
    # Implicit solvent parameters.
    cavity=cavity,
    dielectric=dielectric,
    interactions=interactions)
atoms.calc = calc

# Run the calculation.
opt = BFGS(atoms, trajectory='qn-Pt111-H-sim.traj',
           logfile='qn-Pt111-H-sim.log')
opt.run()

# Tighten the tolerances again.
sj['tol'] = 0.01
sj['always_adjust'] = False
sj['slope'] = None
calc.set(sj=sj)
opt = BFGS(atoms, trajectory='qn-Pt111-H-sim-1.traj',
           logfile='qn-Pt111-H-sim.log')
opt.run()

np.True_

In [7]:
!gdown --fuzzy "https://drive.google.com/file/d/1pTlT-frvB4wIBwG2ZsfDVHrs9BY0vASc/view?usp=drive_link" -O qn-Pt111-H-sim-1.traj
atoms = ase.io.read('qn-Pt111-H-sim-1.traj', index=-1)
view(atoms, viewer='x3d')


Downloading...
From: https://drive.google.com/uc?id=1pTlT-frvB4wIBwG2ZsfDVHrs9BY0vASc
To: /content/qn-Pt111-H-sim-1.traj
100% 5.33k/5.33k [00:00<00:00, 12.3MB/s]


Now we move H to a hollow site instead

In [8]:
# Open the previous simulation and move the H to a hollow site.
z_H=atoms[10].z
atoms[10].position=((atoms[1].position+atoms[2].position+atoms[3].position)/3)
atoms[10].x=(atoms[0].x)
atoms[10].z=z_H
view(atoms, viewer='x3d')

In [ ]:
# Solvated jellium parameters. 20 min
sj = {'target_potential': 4.2,
      'tol': 0.2,
      'always_adjust': True,
      'excess_electrons': -0.01232,  # guess from previous calculation
      'slope': -50.}  # guess from previous calculation

# Implicit solvent parameters (to SolvationGPAW).
epsinf = 78.36  # dielectric constant of water at 298 K
gamma = 18.4 * 1e-3 * Pascal * m
cavity = EffectivePotentialCavity(
    effective_potential=SJMPower12Potential(H2O_layer=False),
    temperature=298.15,  # K
    surface_calculator=GradientSurface())
dielectric = LinearDielectric(epsinf=epsinf)
interactions = [SurfaceInteraction(surface_tension=gamma)]

# The calculator
calc = SJM(
    # General GPAW parameters.
    mode='lcao',
    txt='Pt111-H-hollow.txt',
    gpts=(16, 16, 136),
    kpts=(2, 2, 1),
    xc='PBE',
    maxiter=1000,
    # Solvated jellium parameters.
    sj=sj,
    # Implicit solvent parameters.
    cavity=cavity,
    dielectric=dielectric,
    interactions=interactions)
atoms.calc = calc

# Run the calculation.
opt = BFGS(atoms, trajectory='qn-Pt111-H-hollow.traj',
           logfile='qn-Pt111-H-hollow.log')
opt.run()

# Tighten the tolerances again.
sj['tol'] = 0.01
sj['always_adjust'] = False
sj['slope'] = None
calc.set(sj=sj)
opt = BFGS(atoms, trajectory='qn-Pt111-H-hollow-1.traj',
           logfile='qn-Pt111-H-hollow-1.log')
opt.run()

np.True_

In [9]:
!gdown --fuzzy "https://drive.google.com/file/d/1pTlT-frvB4wIBwG2ZsfDVHrs9BY0vASc/view?usp=drive_link" -O qn-Pt111-H-hollow-1.traj
atoms = ase.io.read('qn-Pt111-H-hollow-1.traj', index=-1)
view(atoms, viewer='x3d')

Downloading...
From: https://drive.google.com/uc?id=1pTlT-frvB4wIBwG2ZsfDVHrs9BY0vASc
To: /content/qn-Pt111-H-hollow-1.traj
100% 5.33k/5.33k [00:00<00:00, 11.5MB/s]


With both the initial and final states converged at the same electrode potential, we can now run the NEB calculation. We create five intermediate images by interpolating between the two endpoints, attach an independent SJM calculator to each image, and optimize the full band with the Dynamic NEB (DyNEB) algorithm. Be aware that this calculation takes more than 12 hours to complete.

In [ ]:
def make_calculator(index):
    # Solvated jellium parameters.
    sj = {'target_potential': 4.2,
          'tol': 0.01,
          'excess_electrons': -0.01232,  # guess from previous
          'slope': -50.}  # guess from previous
    # Implicit solvent parameters (to SolvationGPAW).
    epsinf = 78.36  # dielectric constant of water at 298 K
    gamma = 18.4 * 1e-3 * Pascal * m
    cavity = EffectivePotentialCavity(
        effective_potential=SJMPower12Potential(H2O_layer=False),
        temperature=298.15,  # K
        surface_calculator=GradientSurface())
    dielectric = LinearDielectric(epsinf=epsinf)
    interactions = [SurfaceInteraction(surface_tension=gamma)]
    calc = SJM(
        # General GPAW parameters.
        mode='lcao',
        txt=f'gpaw-{index:d}.txt',
        gpts=(16, 16, 136),
        kpts=(2, 2, 1),
        xc='PBE',
        maxiter=1000,
        # Solvated jellium parameters.
        sj=sj,
        # Implicit solvent parameters.
        cavity=cavity,
        dielectric=dielectric,
        interactions=interactions)
    return calc


# Create the band of images, attaching a calc to each.
initial = ase.io.read('qn-Pt111-H-sim-1.traj')
final = ase.io.read('qn-Pt111-H-hollow-1.traj')
images = [initial]
for index in range(5):
    images += [initial.copy()]
    images[-1].calc = make_calculator(index + 1)
images += [final]
interpolate(images)

# Create and relax the DyNEB.
neb = DyNEB(images)
opt = BFGS(neb, logfile='neb.log', trajectory='neb.traj')
opt.run(fmax=0.05)

/usr/local/lib/python3.12/dist-packages/ase/mep/neb.py:329: UserWarning: The default method has changed from 'aseneb' to 'improvedtangent'. The 'aseneb' method is an unpublished, custom implementation that is not recommended as it frequently results in very poor bands. Please explicitly set method='improvedtangent' to silence this warning, or set method='aseneb' if you strictly require the old behavior (results may vary). See: https://gitlab.com/ase/ase/-/merge_requests/3952
  warnings.warn(


Once the NEB converges, we extract the final images and use NEBTools to plot the energy profile along the minimum-energy path, reading off the activation barrier from the highest-energy image.

In [10]:
!gdown --fuzzy "https://drive.google.com/file/d/1pTlT-frvB4wIBwG2ZsfDVHrs9BY0vASc/view?usp=drive_link" -O neb.traj
atoms = ase.io.read('neb.traj', index=-1)
view(atoms, viewer='x3d')

Downloading...
From: https://drive.google.com/uc?id=1pTlT-frvB4wIBwG2ZsfDVHrs9BY0vASc
To: /content/neb.traj
100% 5.33k/5.33k [00:00<00:00, 18.4MB/s]


In [21]:
from ase.io import read
from ase.mep import NEBTools
from matplotlib import pyplot as plt

images = read('neb.traj@-7:')

print(len(images))   # should be 7


1


In [ ]:
images = ase.io.read('neb.traj', index=':-3')
nebtools = NEBTools(images)
fig = nebtools.plot_band()
pyplot.show()
fig.savefig('band.png')

# Summary

After completing this notebook, you should understand:

1. Why constant-potential NEB matters.

2. How to use a loose SJM tolerance with always_adjust=True for the initial convergence pass, allowing ionic and electron-count updates to proceed simultaneously for speed, then tighten the tolerance and disable always_adjust for the final well-converged structure. Seed the final state optimization with excess_electrons and slope guesses from the converged initial state to accelerate the electron-count search.

You should be able to:

1. Set up initial and final states for a surface reaction. Build the initial state by appending an adsorbate at a specific surface site, and construct the final state by manually relocating the adsorbate to the target site while preserving its z-height.
2. Interpolate and run a NEB band with SJM. Create intermediate images with interpolate(), attach an independent SJM calculator to each image using a factory function, assemble the band with DyNEB, and optimize with BFGS to a force threshold of 0.05 eV/Å.
3. Read the last  frames of the converged NEB trajectory (one per image), pass them to NEBTools, plot the energy profile along the minimum-energy path, and read off the activation barrier as the energy of the highest-energy image relative to the initial state.